# Evaluación DANE — TIC e I+D

## Índice
1. [Parte 1: Evaluación del RAG híbrido](#parte-1-evaluación-del-rag-híbrido) — sin costo, resultados precalculados
2. [Parte 2: Evaluación del agente](#parte-2-evaluación-del-agente) — ejecutar con kernel `.venv`

---

## Parte 1: Evaluación del RAG híbrido

Golden dataset: 50 preguntas documentales. Lee `rag_retrieval_evaluation.json` — sin costo.

**Métricas**
- **Hit Rate@5**: el chunk correcto aparece en los cinco primeros resultados.
- **MRR@5**: premia que el chunk correcto aparezca en posiciones más altas.

### Grid search — combinaciones de pesos E5 / BM25

In [1]:
from pathlib import Path
import json

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent

rag_path = root / 'evaluation' / 'results' / 'rag_retrieval_evaluation.json'
rag = json.loads(rag_path.read_text(encoding='utf-8'))

for row in rag['grid_search']:
    print(f"E5={row['dense_weight']:.1f} | BM25={row['bm25_weight']:.1f} | "
          f"Hit Rate={row['hit_rate']:.3f} | MRR={row['mrr']:.3f}")


E5=0.0 | BM25=1.0 | Hit Rate=0.800 | MRR=0.677
E5=0.2 | BM25=0.8 | Hit Rate=0.860 | MRR=0.699
E5=0.4 | BM25=0.6 | Hit Rate=0.880 | MRR=0.711
E5=0.5 | BM25=0.5 | Hit Rate=0.920 | MRR=0.722
E5=0.6 | BM25=0.4 | Hit Rate=0.920 | MRR=0.707
E5=0.8 | BM25=0.2 | Hit Rate=0.920 | MRR=0.651
E5=1.0 | BM25=0.0 | Hit Rate=0.720 | MRR=0.488


### Posición del chunk correcto en top_k=5

In [1]:
diagnostics = rag['selected_weight_diagnostics']

for rank, count in diagnostics['rank_distribution'].items():
    print(f'{rank}: {count}')


1: 30
2: 8
3: 3
4: 2
5: 3
not_retrieved: 4


### Score híbrido del chunk correcto recuperado

In [1]:
summary = diagnostics['correct_chunk_score_summary']
bins = diagnostics['correct_chunk_score_bins']

for name, value in summary.items():
    print(f'{name}: {value:.3f}' if isinstance(value, float) else f'{name}: {value}')

for interval, count in bins.items():
    print(f'{interval}: {count}')


count: 46
min: 0.753
p25: 0.934
median: 0.967
mean: 0.947
p75: 1.000
max: 1.000

[0.0, 0.2): 0
[0.2, 0.4): 0
[0.4, 0.6): 0
[0.6, 0.8): 3
[0.8, 1.0]: 43


### Conclusión RAG

Usar **E5 = 50%** y **BM25 = 50%**. Hit Rate@5: **0.920**. MRR@5: **0.722**. Validación cruzada anidada (5 folds): Hit Rate **0.920**, MRR **0.722**.

---

## Parte 2: Evaluación del agente

Dataset: 30 preguntas (document_search, statistic_direct, comparison_direct). Reanuda desde `agent_evaluation.json` — solo ejecuta las preguntas pendientes.

**Ejecutar con el entorno de uv. No correr en paralelo con `python -m evaluation.evaluate_agent`.**

### Celda 1 — Imports, rutas y estado del dataset

In [ ]:
import sys
import os
import json
from pathlib import Path
from collections import defaultdict

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from evaluation.agent_helper import (
    load_jsonl, validate_dataset, run_agent, run_judge, save_results, cost_summary
)

GOLDEN = root / 'evaluation' / 'golden_dataset_agent.jsonl'
RESULTS_PATH = root / 'evaluation' / 'results' / 'agent_evaluation.json'

golden = load_jsonl(GOLDEN)
existing = json.loads(RESULTS_PATH.read_text(encoding='utf-8')) if RESULTS_PATH.exists() else []
done_ids = {r['id'] for r in existing}
pending = [r for r in golden if r['id'] not in done_ids]

report = validate_dataset(golden)
print(f"Dataset válido: {report['valid']}")
print(f"Total preguntas: {report['record_count']}")
print(f"Categorías: {report['categories']}")
print(f"Ya ejecutadas: {len(existing)}")
print(f"Pendientes: {len(pending)}")
if pending:
    print("IDs pendientes: " + ", ".join(r['id'] for r in pending))


### Celda 2 — Loop de evaluación (agente + judge por pregunta)

In [ ]:
all_results = list(existing)

for record in pending:
    print(f"Ejecutando {record['id']} ({record['category']})...")
    result = run_agent([record])[0]
    result = run_judge([result])[0]
    all_results.append(result)
    save_results(all_results, RESULTS_PATH)
    print(f"  guardado | agente ${result['agent_cost']:.5f} | judge ${result['judge_cost']:.5f}")

if not pending:
    print("No hay preguntas pendientes.")
else:
    print(f"\nListo. Total guardado: {len(all_results)} preguntas.")


### Celda 3 — Resumen de la corrida actual

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from evaluation.agent_helper import cost_summary

RESULTS_PATH = root / 'evaluation' / 'results' / 'agent_evaluation.json'
results = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
judged = [r for r in results if 'judge' in r]

total = len(judged)
answer_good = sum(r['judge']['answer_score'] == 'good' for r in judged)
trajectory_good = sum(r['judge']['trajectory_score'] == 'good' for r in judged)
sequence_good = sum(r['expected_sequence_used'] for r in judged)
costs = cost_summary(judged)

print(f"Respuestas buenas:   {answer_good}/{total}")
print(f"Trayectorias buenas: {trajectory_good}/{total}")
print(f"Secuencia esperada:  {sequence_good}/{total}")
print()
print(f"Costo agente: ${costs['agent_cost']:.6f}")
print(f"Costo judge:  ${costs['judge_cost']:.6f}")
print(f"Costo total:  ${costs['total_cost']:.6f}")
print()

categories = defaultdict(list)
for r in judged:
    categories[r['category']].append(r)

for cat, rows in sorted(categories.items()):
    a = sum(r['judge']['answer_score'] == 'good' for r in rows)
    t = sum(r['judge']['trajectory_score'] == 'good' for r in rows)
    s = sum(r['expected_sequence_used'] for r in rows)
    print(f"{cat}: respuestas {a}/{len(rows)} | trayectorias {t}/{len(rows)} | secuencia {s}/{len(rows)}")


### Celda 4 — Bitácora de corridas

Compara todas las corridas guardadas en `evaluation/results/`. Los fallbacks se excluyen para mantener comparabilidad entre versiones.

In [ ]:
import json
from pathlib import Path

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent

results_dir = root / 'evaluation' / 'results'

KNOWN_ORDER = [
    'agent_evaluation_baseline.json',
    'agent_evaluation_v1_sin_stopping_rules.json',
    'agent_evaluation_v2_con_stopping_rules_mini.json',
    'agent_evaluation.json',
]

all_files = list(results_dir.glob('agent_evaluation*.json'))
ordered = [results_dir / n for n in KNOWN_ORDER if (results_dir / n).exists()]
for f in sorted(all_files):
    if f not in ordered:
        ordered.append(f)

print(f"{'Corrida':<48} {'Resp':>5} {'Tray':>5} {'Seq':>5} {'Costo':>10}")
print("-" * 78)
for f in ordered:
    data = json.loads(f.read_text(encoding='utf-8'))
    judged = [r for r in data if 'judge' in r and not r.get('category', '').endswith('_fallback')]
    if not judged:
        continue
    n = len(judged)
    ans  = sum(r['judge']['answer_score'] == 'good' for r in judged)
    traj = sum(r['judge']['trajectory_score'] == 'good' for r in judged)
    seq  = sum(r['expected_sequence_used'] for r in judged)
    cost = sum(r.get('total_cost', 0) for r in judged)
    print(f"{f.stem[:47]:<48} {ans}/{n}  {traj}/{n}  {seq}/{n}  ${cost:.4f}")
